# PM2.5 forecasting with a Darts TFTModel

Trains a single global **Temporal Fusion Transformer** across all GISTDA
stations in `clean-data_preprocess_all_stations_daily.csv`, forecasting
`pm25` 1-7 days ahead.

This notebook is the single-file version of the `pm25-tft-model/` project.
Background reading:

- TFT architecture & Darts `TFTModel` API notes: `../REFERENCE.md`
- Full feature-selection rationale + citations: `FEATURE_SELECTION.md`
  (condensed inline below, in Section 2)

Run top to bottom. Section 3 has the editable configuration (paths,
architecture, training loop).

## 1. Setup

```bash
pip install -r requirements.txt
```

(`darts[torch]>=0.46.1,<0.47`, `pandas`, `numpy`, `scikit-learn` - see
`requirements.txt`.)

In [1]:
from __future__ import annotations

import logging
import pickle
import json
from datetime import datetime
import torch
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import (
    Scaler,
    StaticCovariatesTransformer,
)
from darts.models import TFTModel
from sklearn.metrics import mean_absolute_error, r2_score

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("pm25_tft")

W0913 15:18:41.858000 37124 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


## 2. Feature selection (condensed - see `FEATURE_SELECTION.md` for full detail + citations)

The source table has 257 columns: raw weather, hotspot/fire detections,
calendar fields, cyclical encodings, QC flags, imputation flags, lag/
rolling/diff features, and precomputed multi-horizon targets. Most of that
is either pipeline bookkeeping (not a predictor) or redundant with
something else already selected. What's kept, and why:

**Target**: `pm25`.

**Static covariate**: `station_id` only. It already uniquely identifies a
physical location (227 stations), mirroring how the TFT paper uses a
single entity ID (store ID, meter ID) rather than also feeding the
entity's separately-known attributes. Confirmed with the user
(2026-09-12): no lat/long/province on top.

**Known-future covariates** (must be knowable ahead of time - there's no
weather-forecast feed in this dataset, so only calendar-derived signals
qualify): `year_index`, `dow_sin/cos`, `month_sin/cos`, `doy_sin/cos`.
Sine/cosine pairs are used instead of the raw integer calendar columns to
avoid a false discontinuity between adjacent-but-wrapped values (e.g.
December vs. January). **Correction to `feature_manifest.yaml`**: its
`TFT_KNOWN_TIME_FEATURES` view lists `wind_direction_avg_sin/cos` as
known-future, but wind direction is measured weather with no forecast feed
- it was moved to the observed/past-only group instead (confirmed with the
user 2026-09-12).

**Past (observed-only) covariates** - trimmed from the manifest's ~33-column
`TFT_OBSERVED_FEATURES` view down to 14, removing collinear duplicates
(raw vs. log1p, avg/min/max/range, sum/mean/max of the same underlying
quantity) and columns with a documented, cited link to PM2.5:
`temperature_avg`, `temperature_range`, `humidity_avg`, `humidity_range`,
`pressure_avg`, `pressure_range`, `wind_speed_avg`,
`wind_direction_avg_sin/cos`, `log1p_rainfall`, `rain_event`,
`hotspot_present`, `log1p_hotspot_count`, `log1p_hotspot_frp_sum`.

Two data-quality findings from checking the real CSV directly (not just the
schema) shaped this list further:

- **~39 of 227 "stations" have no PM2.5 sensor at all** (pure
  meteorological stations, e.g. *"Narathiwat Weather Observing Station"*) -
  their `pm25` is 90-100% `NaN` for the whole segment. These are excluded
  as training *targets* below (`MAX_TARGET_NAN_FRAC`), not as feature
  columns.
- **`sunshine_duration` was dropped**: despite looking manageable in
  aggregate (~12% missing), it is **100% missing for 80 of 537** otherwise-
  usable station segments - no sensor installed, not scattered gaps.
  Interpolation can't recover a fully-missing column, and imputing a
  fabricated value for 15% of the data would misrepresent real conditions.
- All `*_lag_*`/`*_roll_*`/`*_diff_*` engineered columns are deliberately
  excluded, matching the manifest's own `TFT_OBSERVED_FEATURES` view: the
  TFT's LSTM encoder + attention block is designed to learn that temporal
  structure directly from the raw `input_chunk_length` window (Lim et al.
  2019, §4.3-4.4), so hand-feeding pre-computed lags would just duplicate
  it and blur variable-importance interpretability.

**Forecast horizon**: one multi-step model, `output_chunk_length=7`
(days 1-7 in a single forward pass), confirmed with the user (2026-09-12).
This is also why the precomputed `target_pm25_t1/t2/t3/t7` columns are
unused - Darts' `TFTModel` slices every horizon step out of the `pm25`
series itself.

In [2]:
DATE_COL = "date"

# Used only to group rows into gap-free per-entity series (see Section 4).
# segment_id comes from the preprocessing pipeline and marks where a gap
# longer than a few days was too large to trust as continuous. Neither
# column is a model input.
GROUP_COLS = ["station_id", "segment_id"]

TARGET = "pm25"

# Static covariate: station_id alone uniquely identifies the physical
# location (confirmed with user 2026-09-12 - no lat/long/province).
STATIC_FEATURES = ["station_id"]

# Known-future covariates: deterministically knowable for any date, with no
# dependence on a weather forecast feed that doesn't exist in this dataset.
# wind_direction_*_sin/cos were moved OUT of this group relative to
# feature_manifest.yaml's TFT_KNOWN_TIME_FEATURES view - see Section 2.
FUTURE_FEATURES = [
    "year_index",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
    "doy_sin",
    "doy_cos",
]

# Observed (past-only) covariates: meteorology + fire-activity signals only
# known after the fact. sunshine_duration was checked and dropped (see
# Section 2) - 100% missing for 80/537 real PM2.5-bearing groups.
PAST_FEATURES = [
    "temperature_avg",
    "temperature_range",
    "humidity_avg",
    "humidity_range",
    "pressure_avg",
    "pressure_range",
    "wind_speed_avg",
    "wind_direction_avg_sin",
    "wind_direction_avg_cos",
    "log1p_rainfall",
    "rain_event",
    "hotspot_present",
    "log1p_hotspot_count",
    "log1p_hotspot_frp_sum",
]

# Columns that must be numeric floats (everything read from the CSV as
# object/str due to blank cells needs coercing before TimeSeries creation).
NUMERIC_FEATURES = FUTURE_FEATURES + PAST_FEATURES + [TARGET]

REQUIRED_COLUMNS = [DATE_COL] + GROUP_COLS + STATIC_FEATURES + FUTURE_FEATURES + PAST_FEATURES + [TARGET]

FREQ = "D"


## 3. Configuration

Edit these before running. `DATA_PATH` assumes this notebook lives in
`pm25-tft-model/`, next to the CSV's parent directory.

In [10]:
# Paths work from either the repository root or notebook directory.
PROJECT_DIR = Path.cwd() if Path("FEATURE_SELECTION.md").exists() else Path.cwd() / "pm25-tft-model"
DATA_PATH = PROJECT_DIR.parent / "clean-data_preprocess_all_stations_daily.csv"
OUTPUT_DIR = PROJECT_DIR / "artifacts" / ("r2_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))

VAL_HOLDOUT_DAYS = 60
TEST_HOLDOUT_DAYS = 60
VAL_START_DATE = None
TEST_START_DATE = None
MAX_TARGET_NAN_FRAC = 0.3

INPUT_CHUNK_LENGTH = 60
OUTPUT_CHUNK_LENGTH = 7
HIDDEN_SIZE = 64
LSTM_LAYERS = 1
NUM_ATTENTION_HEADS = 4
DROPOUT = 0.1
HIDDEN_CONTINUOUS_SIZE = 16

# MSE learns a point forecast suitable for squared-error metrics such as R2.
# Per-group training-only standardization prevents large target ranges from
# suppressing gradients and roughly balances errors across station segments.
N_EPOCHS = 15
BATCH_SIZE = 64  # conservative starting point for an 8 GB GPU
LEARNING_RATE = 1e-3
GRADIENT_CLIP_VAL = 1.0
EARLY_STOPPING_PATIENCE = 12
ACCELERATOR = "gpu"
RANDOM_STATE = 42

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required. Select a CUDA-enabled PyTorch kernel and restart. "
                       "See https://pytorch.org/get-started/locally/ for installation.")
print(f"GPU: {torch.cuda.get_device_name(0)}; PyTorch {torch.__version__}; CUDA {torch.version.cuda}")
torch.set_float32_matmul_precision("high")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)


GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU; PyTorch 2.14.0+cu132; CUDA 13.2


## 4. Data preparation

Keep the documented feature set. Fill residual gaps forward only, trim the
leading incomplete rows, and fit standard scalers on training dates only.
All series use float32 for GPU training. Validation includes the final training
encoder window, so its first predicted day is exactly the validation cutoff.
Validation labels stop before the test cutoff. Upstream CSV imputation may
still use future observations; this notebook cannot undo that preprocessing.


In [11]:
@dataclass
class GroupSeries:
    station_id: str
    segment_id: str
    train_target: TimeSeries
    val_target: TimeSeries | None
    full_target: TimeSeries  # full span, unscaled; scaled in _scale_group_series
    past_covariates: TimeSeries  # full span, already scaled
    future_covariates: TimeSeries  # full span, already scaled


@dataclass
class Datasets:
    train_targets: list[TimeSeries]
    train_past_covariates: list[TimeSeries]
    train_future_covariates: list[TimeSeries]
    val_targets: list[TimeSeries]
    val_past_covariates: list[TimeSeries]
    val_future_covariates: list[TimeSeries]
    val_full_targets: list[TimeSeries]
    val_keys: list[tuple[str, str]]
    target_scalers: dict[tuple[str, str], Scaler]
    past_scalers: dict[tuple[str, str], Scaler]
    future_scalers: dict[tuple[str, str], Scaler]
    static_covariates_transformer: StaticCovariatesTransformer
    n_stations: int
    dropped_groups: int
    dropped_no_sensor_groups: int
    dropped_residual_nan_groups: int


def load_raw(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, usecols=REQUIRED_COLUMNS, parse_dates=[DATE_COL])
    df["station_id"] = df["station_id"].astype(str)
    df["segment_id"] = df["segment_id"].astype(str)
    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values(GROUP_COLS + [DATE_COL]).reset_index(drop=True)
    return df


def _has_nan(ts: TimeSeries) -> bool:
    return bool(np.isnan(ts.values(copy=False)).any())


def _fill_gaps(ts: TimeSeries) -> TimeSeries:
    # Never interpolate backward from a future observation.
    values = pd.DataFrame(ts.values(copy=False)).ffill().to_numpy(dtype=np.float32)
    return ts.with_values(values)


In [12]:
def _build_group_series(
    df: pd.DataFrame,
    input_chunk_length: int,
    output_chunk_length: int,
    val_start_date: pd.Timestamp,
    max_target_nan_frac: float,
    test_start_date: pd.Timestamp | None = None,
) -> tuple[list[GroupSeries], int, int, int]:
    min_len = input_chunk_length + output_chunk_length
    groups: list[GroupSeries] = []
    dropped = 0
    dropped_no_sensor = 0
    dropped_residual_nan = 0

    for (station_id, segment_id), g in df.groupby(GROUP_COLS, sort=False):
        g = g.sort_values(DATE_COL)
        if len(g) < min_len:
            dropped += 1
            continue

        # Some station_ids in this dataset are pure meteorological stations
        # ("... Weather Observing Station" / "... Automatic Weather
        # Station") with no PM2.5 sensor at all - their pm25 column is
        # ~90-100% NaN for the whole segment, not just short measurement
        # gaps. MissingValuesFiller cannot interpolate that (nothing to
        # interpolate from), so these must be excluded as targets here,
        # not silently filled.
        if g.loc[g[DATE_COL] < val_start_date, TARGET].isna().mean() > max_target_nan_frac:
            dropped_no_sensor += 1
            continue

        try:
            target_full = TimeSeries.from_dataframe(
                g,
                time_col=DATE_COL,
                value_cols=[TARGET],
                freq=FREQ,
                static_covariates=pd.DataFrame({"station_id": [station_id]}),
            )
            past_full = TimeSeries.from_dataframe(
                g, time_col=DATE_COL, value_cols=PAST_FEATURES, freq=FREQ
            )
            future_full = TimeSeries.from_dataframe(
                g, time_col=DATE_COL, value_cols=FUTURE_FEATURES, freq=FREQ
            )
        except ValueError as exc:
            logger.warning("Skipping station=%s segment=%s: %s", station_id, segment_id, exc)
            dropped += 1
            continue

        target_full = _fill_gaps(target_full)
        past_full = _fill_gaps(past_full)
        future_full = _fill_gaps(future_full)

        # Trim only the leading incomplete prefix; ffill cannot invent a first value.
        complete = np.isfinite(np.concatenate([
            target_full.values(), past_full.values(), future_full.values()
        ], axis=1)).all(axis=1)
        if not complete.any():
            dropped_residual_nan += 1
            continue
        first = int(np.flatnonzero(complete)[0])
        target_full, past_full, future_full = [
            ts[first:].astype(np.float32) for ts in (target_full, past_full, future_full)
        ]
        if any(_has_nan(ts) for ts in (target_full, past_full, future_full)):
            dropped_residual_nan += 1
            continue

        if target_full.end_time() < val_start_date:
            train_target, val_target = target_full, None
        elif target_full.start_time() >= val_start_date:
            train_target, val_target = None, target_full
        else:
            train_target, val_target = target_full.split_before(val_start_date)

        if train_target is None or len(train_target) < min_len:
            dropped += 1
            continue
        if val_target is not None:
            # Include context, but no training labels or test labels in val_loss.
            val_end = target_full.end_time()
            if test_start_date is not None:
                val_end = min(val_end, test_start_date - pd.Timedelta(days=1))
            if val_end < val_start_date + pd.Timedelta(days=output_chunk_length - 1):
                val_target = None
            else:
                val_target = target_full.slice(
                    val_start_date - pd.Timedelta(days=input_chunk_length), val_end
                )

        groups.append(
            GroupSeries(
                station_id=station_id,
                segment_id=segment_id,
                train_target=train_target,
                val_target=val_target,
                full_target=target_full,
                past_covariates=past_full,
                future_covariates=future_full,
            )
        )

    return groups, dropped, dropped_no_sensor, dropped_residual_nan


def _scale_group_series(
    groups: list[GroupSeries], val_start_date: pd.Timestamp
) -> tuple[dict, dict, dict]:
    target_scalers, past_scalers, future_scalers = {}, {}, {}

    for grp in groups:
        key = (grp.station_id, grp.segment_id)

        target_scaler = Scaler(StandardScaler())
        grp.train_target = target_scaler.fit_transform(grp.train_target)
        if grp.val_target is not None:
            grp.val_target = target_scaler.transform(grp.val_target)
        target_scalers[key] = target_scaler
        grp.full_target = target_scaler.transform(grp.full_target)

        # keep_point=False mirrors split_before()'s convention that
        # val_start_date itself belongs to the validation side, so the
        # scaler never sees a validation-period value while fitting.
        # Groups whose whole span ends before val_start_date are all
        # training data; drop_after() would raise because the timestamp
        # lies outside their index, so keep the full series instead.
        # Both covariate blocks share the group's date index.
        ends_before_val = grp.past_covariates.end_time() < val_start_date

        past_train_slice = (
            grp.past_covariates
            if ends_before_val
            else grp.past_covariates.drop_after(val_start_date, keep_point=False)
        )
        if len(past_train_slice) == 0:
            past_train_slice = grp.past_covariates
        past_scaler = Scaler(StandardScaler())
        past_scaler.fit(past_train_slice)
        grp.past_covariates = past_scaler.transform(grp.past_covariates)
        past_scalers[key] = past_scaler

        future_train_slice = (
            grp.future_covariates
            if ends_before_val
            else grp.future_covariates.drop_after(val_start_date, keep_point=False)
        )
        if len(future_train_slice) == 0:
            future_train_slice = grp.future_covariates
        future_scaler = Scaler(StandardScaler())
        future_scaler.fit(future_train_slice)
        grp.future_covariates = future_scaler.transform(grp.future_covariates)
        future_scalers[key] = future_scaler

    return target_scalers, past_scalers, future_scalers


def prepare_datasets(
    csv_path: str,
    input_chunk_length: int,
    output_chunk_length: int,
    val_start_date,
    max_target_nan_frac: float = 0.3,
    test_start_date=None,
) -> Datasets:
    df = load_raw(csv_path)
    val_start_date = pd.Timestamp(val_start_date)

    groups, dropped, dropped_no_sensor, dropped_residual_nan = _build_group_series(
        df, input_chunk_length, output_chunk_length, val_start_date, max_target_nan_frac,
        None if test_start_date is None else pd.Timestamp(test_start_date)
    )
    if not groups:
        raise ValueError(
            "No (station_id, segment_id) group had enough history for "
            f"input_chunk_length={input_chunk_length} + "
            f"output_chunk_length={output_chunk_length}, after excluding "
            "groups without a usable pm25 target."
        )
    logger.info(
        "Built %d group series, dropped %d (too short), %d (no usable pm25 sensor), "
        "%d (unfillable NaN in a fully-missing column).",
        len(groups), dropped, dropped_no_sensor, dropped_residual_nan,
    )

    target_scalers, past_scalers, future_scalers = _scale_group_series(groups, val_start_date)

    train_targets = [grp.train_target for grp in groups]
    static_transformer = StaticCovariatesTransformer()
    train_targets = static_transformer.fit_transform(train_targets)

    val_groups = [grp for grp in groups if grp.val_target is not None]
    if not val_groups:
        raise ValueError("No validation windows; adjust dates or minimum history.")
    val_targets = static_transformer.transform([grp.val_target for grp in val_groups])
    val_keys = [(grp.station_id, grp.segment_id) for grp in val_groups]
    val_full_targets = static_transformer.transform([grp.full_target for grp in val_groups])

    for grp, transformed in zip(groups, train_targets):
        grp.train_target = transformed
    for grp, transformed in zip(val_groups, val_targets):
        grp.val_target = transformed

    # Upper bound on distinct station_id categories the embedding table
    # needs to cover; some of these stations have no usable pm25 sensor and
    # were dropped above, so the encoder may see fewer than this in
    # practice - that's fine, it just leaves a few unused embedding rows.
    n_stations = df["station_id"].nunique()

    return Datasets(
        train_targets=[grp.train_target for grp in groups],
        train_past_covariates=[grp.past_covariates for grp in groups],
        train_future_covariates=[grp.future_covariates for grp in groups],
        val_targets=[grp.val_target for grp in val_groups],
        val_past_covariates=[grp.past_covariates for grp in val_groups],
        val_future_covariates=[grp.future_covariates for grp in val_groups],
        val_full_targets=val_full_targets,
        val_keys=val_keys,
        target_scalers=target_scalers,
        past_scalers=past_scalers,
        future_scalers=future_scalers,
        static_covariates_transformer=static_transformer,
        n_stations=n_stations,
        dropped_groups=dropped,
        dropped_no_sensor_groups=dropped_no_sensor,
        dropped_residual_nan_groups=dropped_residual_nan,
    )

## 5. Build the datasets

In [13]:
_max_date = pd.read_csv(DATA_PATH, usecols=[DATE_COL], parse_dates=[DATE_COL])[DATE_COL].max()
test_start_date = (pd.Timestamp(TEST_START_DATE) if TEST_START_DATE is not None
                   else _max_date - pd.Timedelta(days=TEST_HOLDOUT_DAYS - 1))
val_start_date = (pd.Timestamp(VAL_START_DATE) if VAL_START_DATE is not None
                  else test_start_date - pd.Timedelta(days=VAL_HOLDOUT_DAYS))
if not val_start_date < test_start_date <= _max_date:
    raise ValueError("Require validation start < test start <= data end.")
print(f"Training before {val_start_date.date()}; validation before {test_start_date.date()}; "
      f"test through {_max_date.date()}")
datasets = prepare_datasets(
    csv_path=DATA_PATH, input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH, val_start_date=val_start_date,
    max_target_nan_frac=MAX_TARGET_NAN_FRAC, test_start_date=test_start_date,
)
print(f"Training groups: {len(datasets.train_targets)}; validation groups: {len(datasets.val_targets)}")


C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\63045878.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  else _max_date - pd.Timedelta(days=TEST_HOLDOUT_DAYS - 1))
C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\63045878.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  else test_start_date - pd.Timedelta(days=VAL_HOLDOUT_DAYS))


Training before 2026-05-08; validation before 2026-07-07; test through 2026-09-04


C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\2037174440.py:84: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  val_end = min(val_end, test_start_date - pd.Timedelta(days=1))
C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\2037174440.py:85: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  if val_end < val_start_date + pd.Timedelta(days=output_chunk_length - 1):
C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\2037174440.py:89: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit inst

Training groups: 434; validation groups: 190


## 6. Define and train the `TFTModel`

`categorical_embedding_sizes={"station_id": n_stations}` lets Darts build
the per-station embedding (see `../REFERENCE.md` §3 for the
`StaticCovariatesTransformer` + `categorical_embedding_sizes` pattern this
follows).

In [14]:
model = TFTModel(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    hidden_size=HIDDEN_SIZE,
    lstm_layers=LSTM_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
    categorical_embedding_sizes={"station_id": datasets.n_stations},
    likelihood=None,
    loss_fn=torch.nn.MSELoss(),
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    optimizer_kwargs={"lr": LEARNING_RATE},
    lr_scheduler_cls=torch.optim.lr_scheduler.ReduceLROnPlateau,
    lr_scheduler_kwargs={"mode": "min", "factor": 0.5, "patience": 4, "min_lr": 1e-5},
    random_state=RANDOM_STATE,
    pl_trainer_kwargs={
        "accelerator": ACCELERATOR,
        "devices": [0],
        "precision": "32-true",
        "callbacks": [EarlyStopping(monitor="val_loss", mode="min",
                                    patience=EARLY_STOPPING_PATIENCE, min_delta=1e-4)],
        "gradient_clip_val": GRADIENT_CLIP_VAL,
    },
    add_relative_index=False,
    use_static_covariates=True,
    model_name="pm25_tft",
    save_checkpoints=True,
    work_dir=str(OUTPUT_DIR),
    force_reset=False,
)

val_kwargs = {}
if datasets.val_targets:
    val_kwargs = {
        "val_series": datasets.val_targets,
        "val_past_covariates": datasets.val_past_covariates,
        "val_future_covariates": datasets.val_future_covariates,
    }
else:
    logger.warning("No validation series available; training without a validation set.")


In [15]:
logger.info("Fitting TFTModel for %d epochs ...", N_EPOCHS)
model.fit(
    series=datasets.train_targets,
    past_covariates=datasets.train_past_covariates,
    future_covariates=datasets.train_future_covariates,
    **val_kwargs,
)

2026-09-13 15:21:42,923 INFO Fitting TFTModel for 15 epochs ...
2026-09-13 15:21:42,934 INFO Train dataset contains 343728 samples.
2026-09-13 15:21:42,956 INFO Using user-defined precision: 32-true. The model output will have the same dtype. If you encounter issues, it's usually due to a conflict between input series data type and precision, or an unsupported precision for the given device or model. For more information, see https://github.com/unit8co/darts/pull/2883 for a discussion on low precision options across hardware platforms.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode  | FLOPs
------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\Data\pm25_forcast_tft\ce-kmitl-capstone-project-1\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 12: 100%|██████████| 5371/5371 [06:27<00:00, 13.86it/s, train_loss=0.0266, val_loss=0.204]


TFTModel(output_chunk_shift=0, hidden_size=64, lstm_layers=1, num_attention_heads=4, full_attention=False, feed_forward=GatedResidualNetwork, dropout=0.1, hidden_continuous_size=16, categorical_embedding_sizes={'station_id': 227}, add_relative_index=False, skip_interpolation=False, loss_fn=MSELoss(), likelihood=None, norm_type=LayerNorm, use_static_covariates=True, input_chunk_length=60, output_chunk_length=7, batch_size=64, n_epochs=15, optimizer_kwargs={'lr': 0.001}, lr_scheduler_cls=<class 'torch.optim.lr_scheduler.ReduceLROnPlateau'>, lr_scheduler_kwargs={'mode': 'min', 'factor': 0.5, 'patience': 4, 'min_lr': 1e-05}, random_state=42, pl_trainer_kwargs={'accelerator': 'gpu', 'devices': [0], 'precision': '32-true', 'callbacks': [<pytorch_lightning.callbacks.early_stopping.EarlyStopping object at 0x0000024E6ACD0C20>], 'gradient_clip_val': 1.0}, model_name=pm25_tft, save_checkpoints=True, work_dir=d:\Data\pm25_forcast_tft\ce-kmitl-capstone-project-1\pm25-tft-model\artifacts\r2_20260913

In [16]:
# Load the best checkpoint separately; rerun this cell without retraining.
# map_location loads weights onto CUDA; the saved trainer uses GPU 0.
# Save and evaluate the checkpoint with the lowest validation MSE.
model = TFTModel.load_from_checkpoint(
    model_name="pm25_tft", work_dir=str(OUTPUT_DIR), best=True,
    map_location="cuda:0",
)
model.trainer_params.update({"accelerator": "gpu", "devices": [0]})


2026-09-13 16:54:20,151 INFO loading best-epoch=0-val_loss=0.18.ckpt


## 7. Save artifacts

- `pm25_tft_model.pt` (+ Darts' companion files) - the trained model,
  loadable via `TFTModel.load("artifacts/pm25_tft_model.pt")`.
- `scalers.pkl` - a pickle with `target_scalers`, `past_scalers`,
  `future_scalers` (each a `dict[(station_id, segment_id), Scaler]`) and the
  fitted `static_covariates_transformer`, needed to inverse-transform
  predictions back to µg/m³ and to encode new data consistently at
  inference time.

In [17]:
model_path = OUTPUT_DIR / "pm25_tft_model.pt"
model.save(str(model_path))
logger.info("Saved model to %s", model_path)

scalers_path = OUTPUT_DIR / "scalers.pkl"
with open(scalers_path, "wb") as f:
    pickle.dump(
        {
            "target_scalers": datasets.target_scalers,
            "past_scalers": datasets.past_scalers,
            "future_scalers": datasets.future_scalers,
            "static_covariates_transformer": datasets.static_covariates_transformer,
        },
        f,
    )
logger.info("Saved per-series scalers to %s", scalers_path)

run_config = {
    "objective": "MSE", "target_scaling": "per_group_train_standard_scaler",
    "val_start_date": str(val_start_date), "test_start_date": str(test_start_date),
    "input_chunk_length": INPUT_CHUNK_LENGTH, "output_chunk_length": OUTPUT_CHUNK_LENGTH,
    "hidden_size": HIDDEN_SIZE, "dropout": DROPOUT, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "max_epochs": N_EPOCHS, "seed": RANDOM_STATE,
    "past_features": PAST_FEATURES, "future_features": FUTURE_FEATURES,
    "static_features": STATIC_FEATURES, "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")


2026-09-13 17:01:46,986 INFO Saved model to d:\Data\pm25_forcast_tft\ce-kmitl-capstone-project-1\pm25-tft-model\artifacts\r2_20260913_152121_194893\pm25_tft_model.pt
2026-09-13 17:01:47,006 INFO Saved per-series scalers to d:\Data\pm25_forcast_tft\ce-kmitl-capstone-project-1\pm25-tft-model\artifacts\r2_20260913_152121_194893\scalers.pkl


968

## 8. Rolling validation and held-out test evaluation

Issue a direct seven-day point forecast each day, using only the history
available at that origin. Score each horizon separately in original units,
alongside last-value persistence and weekly persistence on identical rows.
Only complete seven-day forecasts within each split are scored. Exclude
missing/imputed target labels using the source flags; report coverage.
Use validation for development; inspect test only after choices are fixed.
R2 is undefined for constant targets. Negative scores are retained.


In [18]:
# Read source labels separately: never score notebook-filled targets as observations.
label_flags = ["pm25_imputed", "pm25_missing", "pm25_missing_clean", "pm25_missing_original"]
header = pd.read_csv(DATA_PATH, nrows=0).columns
label_flags = [c for c in label_flags if c in header]
labels = pd.read_csv(DATA_PATH, usecols=[DATE_COL] + GROUP_COLS + [TARGET] + label_flags,
                     parse_dates=[DATE_COL], dtype={c: str for c in GROUP_COLS})
valid = labels[TARGET].notna()
for col in label_flags:
    valid &= ~labels[col].astype(str).str.lower().isin(["1", "1.0", "true"])
labels.loc[~valid, TARGET] = np.nan
truth = labels.set_index(GROUP_COLS + [DATE_COL])[TARGET]

def evaluate_window(name, start, end):
    records = []
    for key, full, past, future in zip(
        datasets.val_keys, datasets.val_full_targets,
        datasets.val_past_covariates, datasets.val_future_covariates,
    ):
        stop = min(end, full.end_time())
        first = max(start, full.start_time() + pd.Timedelta(days=INPUT_CHUNK_LENGTH))
        if stop < first + pd.Timedelta(days=OUTPUT_CHUNK_LENGTH - 1):
            continue
        chunks = model.historical_forecasts(
            series=full.slice(full.start_time(), stop),
            past_covariates=past, future_covariates=future,
            forecast_horizon=OUTPUT_CHUNK_LENGTH, stride=1, start=first,
            start_format="value", retrain=False, last_points_only=False,
            overlap_end=False, num_samples=1, verbose=False,
        )
        scaler = datasets.target_scalers[key]
        history = scaler.inverse_transform(full)
        for chunk in chunks:
            predicted = scaler.inverse_transform(chunk)
            origin = predicted.start_time() - pd.Timedelta(days=1)
            persistence = float(history.slice(origin, origin).values().item())
            for horizon, (date, value) in enumerate(zip(predicted.time_index, predicted.values().ravel()), 1):
                weekly_date = date - pd.Timedelta(days=7)
                weekly = float(history.slice(weekly_date, weekly_date).values().item())
                actual = truth.get((*key, date), np.nan)
                records.append(dict(station_id=key[0], segment_id=key[1], date=date,
                                    origin=origin, horizon=horizon, actual=actual,
                                    tft=float(value), persistence=persistence, weekly=weekly))
    predictions = pd.DataFrame(records)
    if predictions.empty:
        raise ValueError(f"No complete forecast windows for {name}.")
    print(f"{name}: observed labels {predictions.actual.notna().sum()} / {len(predictions)} forecast rows")
    rows = []
    for (station, segment, horizon), frame in predictions.groupby(GROUP_COLS + ["horizon"]):
        frame = frame.dropna(subset=["actual"])
        for method in ["tft", "persistence", "weekly"]:
            y, pred = frame.actual.to_numpy(), frame[method].to_numpy()
            rows.append(dict(station_id=station, segment_id=segment, horizon=horizon,
                             method=method, n_points=len(y),
                             r2=r2_score(y, pred) if len(y) >= 2 and np.ptp(y) > 0 else np.nan,
                             mae=mean_absolute_error(y, pred) if len(y) else np.nan,
                             rmse=float(np.sqrt(np.mean((y-pred)**2))) if len(y) else np.nan))
    groups = pd.DataFrame(rows)
    # Preserve segment-wise R2, then give each station equal weight.
    stations = groups.groupby(["station_id", "horizon", "method"])[["r2", "mae", "rmse"]].mean().reset_index()
    summary = stations.groupby(["horizon", "method"]).agg(
        mean_r2=("r2", "mean"), median_r2=("r2", "median"),
        mean_mae=("mae", "mean"), mean_rmse=("rmse", "mean"),
        stations_with_r2=("r2", "count"),
    )
    print(summary.to_string())
    for suffix, frame in [("predictions", predictions), ("groups", groups),
                          ("stations", stations), ("summary", summary.reset_index())]:
        frame.to_csv(OUTPUT_DIR / f"{name}_{suffix}.csv", index=False)
    return predictions, groups, stations, summary

val_results = evaluate_window("validation", val_start_date, test_start_date - pd.Timedelta(days=1))


C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\1199442173.py:71: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  val_results = evaluate_window("validation", val_start_date, test_start_date - pd.Timedelta(days=1))
C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\1199442173.py:20: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  first = max(start, full.start_time() + pd.Timedelta(days=INPUT_CHUNK_LENGTH))
C:\Users\Tawan-Labtop\AppData\Local\Temp\ipykernel_37124\1199442173.py:21: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.

validation: observed labels 69125 / 69125 forecast rows
                      mean_r2  median_r2  mean_mae  mean_rmse  stations_with_r2
horizon method                                                                 
1       persistence  0.172332   0.231888  2.314316   3.106959               190
        tft         -0.207314   0.052016  3.011758   3.732439               190
        weekly      -1.975505  -1.214895  4.362321   5.935743               190
2       persistence -0.417394  -0.464306  3.051207   4.065340               190
        tft         -0.595161  -0.169482  3.386452   4.190663               190
        weekly      -1.994665  -1.294372  4.332676   5.901031               190
3       persistence -0.864915  -0.877394  3.390778   4.552615               190
        tft         -0.858605  -0.270219  3.498274   4.353161               190
        weekly      -1.992953  -1.676802  4.222786   5.768586               190
4       persistence -1.213768  -1.056166  3.578003   4.809275   

## 9. Final test (run after fixing your training choices)

The test period never contributes to scaler fitting, training labels, early
stopping, or learning-rate scheduling. Earlier observed test days may enter
later encoder windows: this is a rolling forecast with daily updates.

The feature set is unchanged. Larger capacity and MSE are hypotheses to test,
not a guarantee of higher R2. Standardized MSE is a surrogate for station-level
R2, not the exact macro-R2 objective. This point model does not produce
quantile intervals. Results use a new split and observed labels, so they are
not directly comparable with the old validation score. The upstream CSV's
imputation must be audited before claiming a fully leakage-free experiment.


In [ ]:
test_results = evaluate_window("test", test_start_date, _max_date)
